# True Grind — YOLOv8 Particle Segmentation Training

This notebook downloads 5 particle/grain datasets from Roboflow, merges them into one training set, and trains YOLOv8n-Seg for coffee particle segmentation.

**Before running:**
1. Go to **Runtime → Change runtime type → GPU** (pick A100 or T4)
2. Have your Roboflow API key ready (from app.roboflow.com → Settings → API)

## Step 1: Install dependencies

In [ ]:
!pip install -q ultralytics roboflow

## Step 2: Enter your Roboflow API key

Run this cell — it will ask you to paste your key (hidden input).

In [ ]:
from getpass import getpass
ROBOFLOW_API_KEY = getpass("Paste your Roboflow API key: ")
print(f"Key set: {ROBOFLOW_API_KEY[:6]}...")

## Step 3: Check GPU

Make sure you have a GPU assigned.

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected! Go to Runtime → Change runtime type → GPU")

## Step 4: Download all 5 datasets from Roboflow

In [ ]:
from roboflow import Roboflow
from pathlib import Path
import shutil

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

DATASETS = [
    ("winxos-16ly4",      "grain-3z0bc",                    11, "yolov8"),
    ("particle-training", "particle-segmentation-v2-wxihe", 12, "yolov8"),
    ("seed-mrn2d",        "seed-instance-segmentation",      5, "yolov8"),
    ("kopi-otjpe",        "coffee-bean-ztxe3",               2, "yolov8"),
    ("sagmentrice",       "rice-grain-segmentation-jcaab",   8, "yolov8"),
]

RAW_DIR = Path("raw_datasets")
RAW_DIR.mkdir(exist_ok=True)

roots = []
for workspace, project_name, version, fmt in DATASETS:
    dest = RAW_DIR / project_name
    if dest.exists():
        print(f"✓ {project_name} already downloaded")
        roots.append(dest)
        continue
    print(f"Downloading {workspace}/{project_name} v{version}...")
    try:
        project = rf.workspace(workspace).project(project_name)
        project.version(version).download(fmt, location=str(dest))
        roots.append(dest)
        print(f"  ✓ Saved to {dest}")
    except Exception as e:
        print(f"  ✗ Skipped {project_name}: {e}")

print(f"\n{len(roots)} datasets downloaded.")

## Step 5: Merge datasets and remap all labels to 'particle'

In [ ]:
import random

OUT_DIR = Path("data")
VAL_SPLIT = 0.15


def remap_label_file(src, dst):
    """Remap all class IDs to 0 (particle)."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    lines = src.read_text().strip().splitlines()
    remapped = []
    for line in lines:
        if not line.strip():
            continue
        parts = line.split()
        parts[0] = "0"
        remapped.append(" ".join(parts))
    dst.write_text("\n".join(remapped) + "\n" if remapped else "")


def collect_pairs(root):
    """Collect all (image, label) pairs from a Roboflow download."""
    pairs = []
    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    for split in ["train", "valid", "test", ""]:
        img_dir = root / split / "images" if split else root / "images"
        lbl_dir = root / split / "labels" if split else root / "labels"
        if not img_dir.exists():
            continue
        for img in sorted(img_dir.iterdir()):
            if img.suffix.lower() not in img_exts:
                continue
            lbl = lbl_dir / (img.stem + ".txt")
            pairs.append((img, lbl if lbl.exists() else None))
    return pairs


# Clean and rebuild
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)

all_pairs = []
for root in roots:
    pairs = collect_pairs(root)
    print(f"{root.name}: {len(pairs)} images")
    all_pairs.extend(pairs)

print(f"\nTotal: {len(all_pairs)} images")

random.seed(42)
random.shuffle(all_pairs)

n_val = max(1, int(len(all_pairs) * VAL_SPLIT))
val_set = all_pairs[:n_val]
train_set = all_pairs[n_val:]

print(f"Train: {len(train_set)}, Val: {len(val_set)}")

for split_name, split_pairs in [("train", train_set), ("val", val_set)]:
    img_out = OUT_DIR / "images" / split_name
    lbl_out = OUT_DIR / "labels" / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img_path, lbl_path in split_pairs:
        stem = f"{img_path.parent.parent.parent.name}__{img_path.stem}"
        shutil.copy2(img_path, img_out / (stem + img_path.suffix))
        dst_lbl = lbl_out / (stem + ".txt")
        if lbl_path:
            remap_label_file(lbl_path, dst_lbl)
        else:
            dst_lbl.write_text("")

# Write data.yaml
yaml_content = f"""path: {OUT_DIR.resolve()}
train: images/train
val: images/val
nc: 1
names: ['particle']
"""
(OUT_DIR / "data.yaml").write_text(yaml_content)
print(f"\n✅ Dataset ready at {OUT_DIR}/")
print(f"   data.yaml written to {OUT_DIR / 'data.yaml'}")

## Step 6: Train YOLOv8n-Seg

This is the main training cell. Should take **30-60 min on A100**, **1-2 hours on T4**.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")

results = model.train(
    data="data/data.yaml",
    epochs=50,
    imgsz=640,
    project="grind_model",
    name="particle_seg",
    device=0,              # GPU 0
    batch=16,
    workers=4,
    patience=15,           # early stopping
    augment=True,
    hsv_h=0.01,            # minimal hue shift (grounds are brown)
    hsv_s=0.3,
    hsv_v=0.3,
    fliplr=0.5,
    flipud=0.2,
    degrees=15,
    conf=0.25,
    max_det=500,
    nms=True,
)

print("\n✅ Training complete!")
print(f"Best weights: grind_model/particle_seg/weights/best.pt")

## Step 7: Check results

In [ ]:
from IPython.display import Image, display
import os

results_dir = "grind_model/particle_seg"

# Show training curves
if os.path.exists(f"{results_dir}/results.png"):
    display(Image(f"{results_dir}/results.png", width=800))

# Show confusion matrix
if os.path.exists(f"{results_dir}/confusion_matrix.png"):
    display(Image(f"{results_dir}/confusion_matrix.png", width=500))

# Show validation predictions
if os.path.exists(f"{results_dir}/val_batch0_pred.jpg"):
    display(Image(f"{results_dir}/val_batch0_pred.jpg", width=800))

# Print final metrics
import csv
with open(f"{results_dir}/results.csv") as f:
    rows = list(csv.reader(f))
    header = [h.strip() for h in rows[0]]
    last = [v.strip() for v in rows[-1]]
    print("\nFinal epoch metrics:")
    for h, v in zip(header, last):
        print(f"  {h}: {v}")

## Step 8: Download the trained weights

Run this to download `best.pt` to your computer. Then put it in your `truegrind-final/` folder.

In [ ]:
from google.colab import files

weights_path = "grind_model/particle_seg/weights/best.pt"
print(f"Downloading {weights_path}...")
files.download(weights_path)

## Done! 🎉

**Next steps:**
1. Move `best.pt` into your `truegrind-final/` folder on your Mac
2. Test it: `.venv/bin/python grind_pipeline.py infer your_photo.jpg --model best.pt`
3. Take photos of your Fellow Ode grounds, label in Roboflow, and run Stage 2 fine-tuning